In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(r"C:\Users\aduak\Downloads\capstone")
CLEAN_DIR = DATA_DIR / "cleaned"

cweeds = pd.read_csv(
    CLEAN_DIR / "clean_cweeds_4cities.csv",
    parse_dates=["timestamp"],
    low_memory=False
)

print("Shape:", cweeds.shape)
print(cweeds["city"].value_counts())

Shape: (701280, 64)
city
Calgary         175320
Edmonton        175320
Lethbridge      175320
Medicine Hat    175320
Name: count, dtype: int64


In [3]:
tilt_variables = [
    "ghi_wh_m2",
    "dni_wh_m2",
    "dhi_wh_m2"
]

tilt_data_quality = (
    cweeds
    .groupby("city")[tilt_variables]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

display(tilt_data_quality)

ghi_wh_m2                      dni_wh_m2                        \
                 count    mean  min     max     count    mean  min      max   
city                                                                          
Calgary         175320  150.77  0.0  972.22    175320  187.67  0.0  1022.78   
Edmonton        175320  142.57  0.0  952.50    175320  174.09  0.0  1010.00   
Lethbridge      175320  159.28  0.0  976.67    175320  196.52  0.0  1030.56   
Medicine Hat    175320  157.29  0.0  974.72    175320  191.35  0.0  1028.33   

             dhi_wh_m2                      
                 count   mean  min     max  
city                                        
Calgary         175320  59.22  0.0  532.22  
Edmonton        175320  58.73  0.0  423.89  
Lethbridge      175320  58.69  0.0  448.33  
Medicine Hat    175320  59.27  0.0  667.22

In [4]:
import pvlib

print("pvlib version:", pvlib.__version__)

pvlib version: 0.15.2


In [5]:
tilt_df = cweeds[
    [
        "timestamp",
        "city",
        "latitude",
        "longitude",
        "elevation_m",
        "utc_offset",
        "ghi_wh_m2",
        "dni_wh_m2",
        "dhi_wh_m2"
    ]
].copy()

display(tilt_df.head())

,timestamp,city,latitude,longitude,elevation_m,utc_offset,ghi_wh_m2,dni_wh_m2,dhi_wh_m2
0,1998-01-01 00:00:00,Calgary,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0
1,1998-01-01 01:00:00,Calgary,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0
2,1998-01-01 02:00:00,Calgary,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0
3,1998-01-01 03:00:00,Calgary,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0
4,1998-01-01 04:00:00,Calgary,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0


In [6]:
# CWEEDS timestamps are local standard time.
# Alberta standard time is UTC-7.

tilt_df["timestamp_local"] = (
    tilt_df["timestamp"]
    .dt.tz_localize("Etc/GMT+7")
)

print(tilt_df["timestamp_local"].head())

0   1998-01-01 00:00:00-07:00
1   1998-01-01 01:00:00-07:00
2   1998-01-01 02:00:00-07:00
3   1998-01-01 03:00:00-07:00
4   1998-01-01 04:00:00-07:00
Name: timestamp_local, dtype: datetime64[us, Etc/GMT+7]


In [7]:
solar_position_parts = []

for city, group in tilt_df.groupby("city"):

    print("Calculating solar position for:", city)

    group = group.copy()

    latitude = group["latitude"].iloc[0]
    longitude = group["longitude"].iloc[0]
    elevation = group["elevation_m"].iloc[0]

    solar_position = pvlib.solarposition.get_solarposition(
        time=group["timestamp_local"],
        latitude=latitude,
        longitude=longitude,
        altitude=elevation
    )

    group["solar_zenith"] = solar_position["apparent_zenith"].values
    group["solar_azimuth"] = solar_position["azimuth"].values

    solar_position_parts.append(group)

tilt_solar = pd.concat(
    solar_position_parts,
    ignore_index=True
)

print("Solar position calculated.")

display(
    tilt_solar[
        [
            "timestamp",
            "city",
            "solar_zenith",
            "solar_azimuth",
            "ghi_wh_m2"
        ]
    ].head(24)
)

Calculating solar position for: Calgary
Calculating solar position for: Edmonton
Calculating solar position for: Lethbridge
Calculating solar position for: Medicine Hat
Solar position calculated.


,timestamp,city,solar_zenith,solar_azimuth,ghi_wh_m2
0,1998-01-01 00:00:00,Calgary,150.876069,341.073294,0.000000
1,1998-01-01 01:00:00,Calgary,151.618146,9.948883,0.000000
2,1998-01-01 02:00:00,Calgary,147.868734,36.523086,0.000000
3,1998-01-01 03:00:00,Calgary,140.978249,57.228074,0.000000
4,1998-01-01 04:00:00,Calgary,132.437187,73.121121,0.000000
5,1998-01-01 05:00:00,Calgary,123.189192,86.132067,0.000000
6,1998-01-01 06:00:00,Calgary,113.793944,97.658069,0.000000
7,1998-01-01 07:00:00,Calgary,104.638002,108.607518,0.000000
8,1998-01-01 08:00:00,Calgary,96.053424,119.600966,2.222222
9,1998-01-01 09:00:00,Calgary,88.108757,131.086134,33.888889


In [8]:
daytime_geometry = (
    tilt_solar[
        tilt_solar["ghi_wh_m2"] > 0
    ]
    .groupby("city")
    .agg(
        min_zenith=("solar_zenith", "min"),
        max_daylight_zenith=("solar_zenith", "max"),
        min_azimuth=("solar_azimuth", "min"),
        max_azimuth=("solar_azimuth", "max")
    )
    .round(2)
)

display(daytime_geometry)

,min_zenith,max_daylight_zenith,min_azimuth,max_azimuth
city,,,,
Calgary,27.99,99.24,44.93,300.42
Edmonton,30.20,98.71,45.45,300.55
Lethbridge,26.78,99.40,46.01,301.45
Medicine Hat,27.01,148.17,47.76,320.14


In [9]:
tilt_angles = np.arange(
    20,
    61,
    1
)

print("Number of tilt angles:", len(tilt_angles))
print(tilt_angles)

Number of tilt angles: 41
[20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43
 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60]


In [10]:
surface_azimuth = 180

In [11]:
tilt_results = []

GROUND_ALBEDO = 0.20

for city, group in tilt_solar.groupby("city"):

    print("Simulating:", city)

    for tilt in tilt_angles:

        poa = pvlib.irradiance.get_total_irradiance(
            surface_tilt=tilt,
            surface_azimuth=180,

            solar_zenith=group["solar_zenith"],
            solar_azimuth=group["solar_azimuth"],

            dni=group["dni_wh_m2"],
            ghi=group["ghi_wh_m2"],
            dhi=group["dhi_wh_m2"],

            albedo=GROUND_ALBEDO,

            model="isotropic"
        )

        # Prevent tiny numerical negatives
        poa_global = poa["poa_global"].clip(lower=0)

        tilt_results.append({
            "City": city,
            "Tilt_Degrees": tilt,
            "Total_POA_Wh_m2": poa_global.sum(),
            "Mean_POA_Wh_m2": poa_global.mean()
        })

tilt_results = pd.DataFrame(tilt_results)

display(tilt_results.head(15))

Simulating: Calgary
Simulating: Edmonton
Simulating: Lethbridge
Simulating: Medicine Hat


,City,Tilt_Degrees,Total_POA_Wh_m2,Mean_POA_Wh_m2
0,Calgary,20,3.084467e+07,175.933530
1,Calgary,21,3.099388e+07,176.784597
2,Calgary,22,3.113609e+07,177.595743
3,Calgary,23,3.127126e+07,178.366745
4,Calgary,24,3.139938e+07,179.097557
5,Calgary,25,3.152047e+07,179.788197
6,Calgary,26,3.163449e+07,180.438561
7,Calgary,27,3.174142e+07,181.048461
8,Calgary,28,3.184127e+07,181.618009
9,Calgary,29,3.193407e+07,182.147356


In [12]:
optimal_tilt = (
    tilt_results.loc[
        tilt_results
        .groupby("City")["Total_POA_Wh_m2"]
        .idxmax()
    ]
    .copy()
    .sort_values("City")
    .reset_index(drop=True)
)

optimal_tilt["Annual_POA_kWh_m2"] = (
    optimal_tilt["Total_POA_Wh_m2"]
    / 20
    / 1000
)

display(
    optimal_tilt[
        [
            "City",
            "Tilt_Degrees",
            "Annual_POA_kWh_m2"
        ]
    ].round(2)
)

,City,Tilt_Degrees,Annual_POA_kWh_m2
0,Calgary,42,1624.91
1,Edmonton,42,1537.00
2,Lethbridge,40,1687.64
3,Medicine Hat,39,1659.30


In [13]:
city_locations = (
    tilt_solar
    .groupby("city")
    .agg(
        latitude=("latitude", "first")
    )
    .reset_index()
)

city_locations["Formula_Tilt"] = (
    city_locations["latitude"] * 0.76 + 3.1
)

city_locations["Formula_Tilt"] = (
    city_locations["Formula_Tilt"]
    .round(1)
)

tilt_comparison = pd.merge(
    optimal_tilt[
        ["City", "Tilt_Degrees", "Annual_POA_kWh_m2"]
    ],
    city_locations,
    left_on="City",
    right_on="city",
    how="left"
)

tilt_comparison["Difference_Degrees"] = (
    tilt_comparison["Tilt_Degrees"] -
    tilt_comparison["Formula_Tilt"]
)

tilt_comparison = tilt_comparison[
    [
        "City",
        "latitude",
        "Formula_Tilt",
        "Tilt_Degrees",
        "Difference_Degrees",
        "Annual_POA_kWh_m2"
    ]
]

display(tilt_comparison.round(2))

,City,latitude,Formula_Tilt,Tilt_Degrees,Difference_Degrees,Annual_POA_kWh_m2
0,Calgary,51.11,41.9,42,0.1,1624.91
1,Edmonton,53.31,43.6,42,-1.6,1537.00
2,Lethbridge,49.70,40.9,40,-0.9,1687.64
3,Medicine Hat,50.03,41.1,39,-2.1,1659.30


In [14]:
horizontal_summary = (
    cweeds
    .groupby("city")
    .agg(
        Annual_Horizontal_kWh_m2=(
            "ghi_wh_m2",
            lambda x: x.sum() / 20 / 1000
        )
    )
    .reset_index()
)

tilt_gain = pd.merge(
    tilt_comparison,
    horizontal_summary,
    left_on="City",
    right_on="city",
    how="left"
)

tilt_gain["Tilt_Gain_Percent"] = (
    (
        tilt_gain["Annual_POA_kWh_m2"]
        - tilt_gain["Annual_Horizontal_kWh_m2"]
    )
    / tilt_gain["Annual_Horizontal_kWh_m2"]
    * 100
)

display(
    tilt_gain[
        [
            "City",
            "Tilt_Degrees",
            "Annual_Horizontal_kWh_m2",
            "Annual_POA_kWh_m2",
            "Tilt_Gain_Percent"
        ]
    ].round(2)
)

,City,Tilt_Degrees,Annual_Horizontal_kWh_m2,Annual_POA_kWh_m2,Tilt_Gain_Percent
0,Calgary,42,1321.63,1624.91,22.95
1,Edmonton,42,1249.79,1537.00,22.98
2,Lethbridge,40,1396.24,1687.64,20.87
3,Medicine Hat,39,1378.79,1659.30,20.34


### Annual PV Energy ≈ Annual POA × System Size × Performance Ratio ###

In [15]:
PERFORMANCE_RATIO = 0.80

system_sizes_kwp = [10, 100]

pv_yield_results = []

for _, row in tilt_comparison.iterrows():

    for system_size in system_sizes_kwp:

        annual_energy_kwh = (
            row["Annual_POA_kWh_m2"]
            * system_size
            * PERFORMANCE_RATIO
        )

        pv_yield_results.append({
            "City": row["City"],
            "Optimal_Tilt_Deg": row["Tilt_Degrees"],
            "System_Size_kWp": system_size,
            "Performance_Ratio": PERFORMANCE_RATIO,
            "Estimated_Annual_Energy_kWh": annual_energy_kwh
        })

pv_yield_results = pd.DataFrame(pv_yield_results)

display(pv_yield_results.round(2))

,City,Optimal_Tilt_Deg,System_Size_kWp,Performance_Ratio,Estimated_Annual_Energy_kWh
0,Calgary,42,10,0.8,12999.24
1,Calgary,42,100,0.8,129992.42
2,Edmonton,42,10,0.8,12295.97
3,Edmonton,42,100,0.8,122959.71
4,Lethbridge,40,10,0.8,13501.16
5,Lethbridge,40,100,0.8,135011.59
6,Medicine Hat,39,10,0.8,13274.41
7,Medicine Hat,39,100,0.8,132744.10


In [17]:
# Recreate tilt_summary from the previously calculated tilt results

tilt_summary = pd.DataFrame({
    "City": ["Calgary", "Edmonton", "Lethbridge", "Medicine Hat"],
    "latitude": [51.11, 53.31, 49.70, 50.03],
    "Formula_Tilt": [41.9, 43.6, 40.9, 41.1],
    "Tilt_Degrees": [42, 42, 40, 39],
    "Difference_Degrees": [0.1, -1.6, -0.9, -2.1],
    "Annual_POA_kWh_m2": [1624.91, 1537.00, 1687.64, 1659.30]
})

display(tilt_summary)

,City,latitude,Formula_Tilt,Tilt_Degrees,Difference_Degrees,Annual_POA_kWh_m2
0,Calgary,51.11,41.9,42,0.1,1624.91
1,Edmonton,53.31,43.6,42,-1.6,1537.00
2,Lethbridge,49.70,40.9,40,-0.9,1687.64
3,Medicine Hat,50.03,41.1,39,-2.1,1659.30


In [18]:
# ============================================================
# PV SYSTEM SIZING AND ANNUAL GENERATION
# ============================================================

PR = 0.80

RESIDENTIAL_KWP = 10
INDUSTRIAL_KWP = 100

pv_results = tilt_summary.copy()

# Annual generation:
# POA irradiance (kWh/m²/year) × system capacity (kWp) × PR

pv_results["Residential_10kWp_kWh"] = (
    pv_results["Annual_POA_kWh_m2"]
    * RESIDENTIAL_KWP
    * PR
)

pv_results["Industrial_100kWp_kWh"] = (
    pv_results["Annual_POA_kWh_m2"]
    * INDUSTRIAL_KWP
    * PR
)

pv_results = pv_results[
    [
        "City",
        "Tilt_Degrees",
        "Annual_POA_kWh_m2",
        "Residential_10kWp_kWh",
        "Industrial_100kWp_kWh"
    ]
].round(2)

display(pv_results)

,City,Tilt_Degrees,Annual_POA_kWh_m2,Residential_10kWp_kWh,Industrial_100kWp_kWh
0,Calgary,42,1624.91,12999.28,129992.8
1,Edmonton,42,1537.00,12296.00,122960.0
2,Lethbridge,40,1687.64,13501.12,135011.2
3,Medicine Hat,39,1659.30,13274.40,132744.0


In [19]:
# ============================================================
# SIMPLE PAYBACK ANALYSIS
# ============================================================

ELECTRICITY_RATE = 0.14      # $/kWh
INSTALLED_COST_PER_WP = 2.50 # $/Wp

# Installed system costs
RESIDENTIAL_COST = RESIDENTIAL_KWP * 1000 * INSTALLED_COST_PER_WP
INDUSTRIAL_COST = INDUSTRIAL_KWP * 1000 * INSTALLED_COST_PER_WP

payback_results = pv_results.copy()

# Annual value of generated electricity
payback_results["Residential_Annual_Savings_$"] = (
    payback_results["Residential_10kWp_kWh"] * ELECTRICITY_RATE
)

payback_results["Industrial_Annual_Savings_$"] = (
    payback_results["Industrial_100kWp_kWh"] * ELECTRICITY_RATE
)

# Simple payback period
payback_results["Residential_Payback_Years"] = (
    RESIDENTIAL_COST /
    payback_results["Residential_Annual_Savings_$"]
)

payback_results["Industrial_Payback_Years"] = (
    INDUSTRIAL_COST /
    payback_results["Industrial_Annual_Savings_$"]
)

display(
    payback_results[
        [
            "City",
            "Tilt_Degrees",
            "Residential_10kWp_kWh",
            "Residential_Annual_Savings_$",
            "Residential_Payback_Years",
            "Industrial_100kWp_kWh",
            "Industrial_Annual_Savings_$",
            "Industrial_Payback_Years"
        ]
    ].round(2)
)

,City,Tilt_Degrees,Residential_10kWp_kWh,Residential_Annual_Savings_$,Residential_Payback_Years,Industrial_100kWp_kWh,Industrial_Annual_Savings_$,Industrial_Payback_Years
0,Calgary,42,12999.28,1819.90,13.74,129992.8,18198.99,13.74
1,Edmonton,42,12296.00,1721.44,14.52,122960.0,17214.40,14.52
2,Lethbridge,40,13501.12,1890.16,13.23,135011.2,18901.57,13.23
3,Medicine Hat,39,13274.40,1858.42,13.45,132744.0,18584.16,13.45


In [20]:
# ============================================================
# PAYBACK SENSITIVITY ANALYSIS
# ============================================================

scenarios = pd.DataFrame({
    "Scenario": ["Low", "Base", "High"],
    "Installed_Cost_per_Wp": [2.00, 2.50, 3.00],
    "Electricity_Rate_per_kWh": [0.16, 0.14, 0.12]
})

sensitivity_rows = []

for _, city_row in pv_results.iterrows():

    for _, scenario in scenarios.iterrows():

        # Residential 10 kWp
        residential_cost = (
            RESIDENTIAL_KWP
            * 1000
            * scenario["Installed_Cost_per_Wp"]
        )

        residential_savings = (
            city_row["Residential_10kWp_kWh"]
            * scenario["Electricity_Rate_per_kWh"]
        )

        residential_payback = (
            residential_cost / residential_savings
        )

        # Industrial 100 kWp
        industrial_cost = (
            INDUSTRIAL_KWP
            * 1000
            * scenario["Installed_Cost_per_Wp"]
        )

        industrial_savings = (
            city_row["Industrial_100kWp_kWh"]
            * scenario["Electricity_Rate_per_kWh"]
        )

        industrial_payback = (
            industrial_cost / industrial_savings
        )

        sensitivity_rows.append({
            "City": city_row["City"],
            "Scenario": scenario["Scenario"],
            "Installed_Cost_per_Wp": scenario["Installed_Cost_per_Wp"],
            "Electricity_Rate_per_kWh": scenario["Electricity_Rate_per_kWh"],
            "Residential_Payback_Years": residential_payback,
            "Industrial_Payback_Years": industrial_payback
        })

payback_sensitivity = pd.DataFrame(sensitivity_rows)

display(
    payback_sensitivity.round(2)
)

,City,Scenario,Installed_Cost_per_Wp,Electricity_Rate_per_kWh,Residential_Payback_Years,Industrial_Payback_Years
0,Calgary,Low,2.0,0.16,9.62,9.62
1,Calgary,Base,2.5,0.14,13.74,13.74
2,Calgary,High,3.0,0.12,19.23,19.23
3,Edmonton,Low,2.0,0.16,10.17,10.17
4,Edmonton,Base,2.5,0.14,14.52,14.52
5,Edmonton,High,3.0,0.12,20.33,20.33
6,Lethbridge,Low,2.0,0.16,9.26,9.26
7,Lethbridge,Base,2.5,0.14,13.23,13.23
8,Lethbridge,High,3.0,0.12,18.52,18.52
9,Medicine Hat,Low,2.0,0.16,9.42,9.42


In [21]:
payback_range = (
    payback_sensitivity
    .groupby("City")
    .agg(
        Best_Case_Payback_Years=("Residential_Payback_Years", "min"),
        Base_Case_Payback_Years=("Residential_Payback_Years", "median"),
        Worst_Case_Payback_Years=("Residential_Payback_Years", "max")
    )
    .round(2)
    .reset_index()
)

display(payback_range)

,City,Best_Case_Payback_Years,Base_Case_Payback_Years,Worst_Case_Payback_Years
0,Calgary,9.62,13.74,19.23
1,Edmonton,10.17,14.52,20.33
2,Lethbridge,9.26,13.23,18.52
3,Medicine Hat,9.42,13.45,18.83


In [22]:
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

tilt_comparison.to_csv(
    OUTPUT_DIR / "tilt_optimization_by_city.csv",
    index=False
)

pv_results.to_csv(
    OUTPUT_DIR / "pv_yield_10kwp_100kwp.csv",
    index=False
)

payback_results.to_csv(
    OUTPUT_DIR / "base_case_payback_by_city.csv",
    index=False
)

payback_sensitivity.to_csv(
    OUTPUT_DIR / "payback_sensitivity_all_scenarios.csv",
    index=False
)

payback_range.to_csv(
    OUTPUT_DIR / "payback_range_by_city.csv",
    index=False
)

print("D3 optimization outputs saved.")

D3 optimization outputs saved.
